# Streaming Bronze — Kafka to Parquet

Consumes the `healthcare_fhir_stream` Kafka topic (one message per FHIR resource,
published by `scripts/02_kafka_producer.py`) and lands it in the bronze layer.

Output schema matches `11_batch_processing_bronze.ipynb` exactly, so the silver
notebooks can read batch and streaming bronze the same way. Streaming output is
written to a **separate path** (`bronze/streaming_data/`) from batch.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, MapType
import glob

In [2]:
try:
    spark.stop()
except:
    pass

## Spark session

The Kafka connector is **not** bundled with Spark, so its jars must be on the classpath
before the session starts — you cannot add them afterwards.

We point `spark.jars` at the four jars vendored in `healthcare_de_project/jars/` rather
than using `spark.jars.packages`. The jar versions must match the Spark build in
`SPARK_HOME` (4.0.1 here) — a mismatched connector fails at task time with confusing
errors like `NoSuchFieldError`.

In [3]:
kafka_jars = ",".join(sorted(glob.glob("../../jars/*.jar")))
print("jars on classpath:")
for j in sorted(glob.glob("../../jars/*.jar")):
    print("  ", j.split("/")[-1])

spark = (SparkSession.builder.appName("HealthcareStreamingBronze")
    .config("spark.jars", kafka_jars)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .master("local[*]")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("\nspark version:", spark.version)

jars on classpath:
   org.apache.commons_commons-pool2-2.12.1.jar
   org.apache.kafka_kafka-clients-3.9.1.jar
   spark-sql-kafka-0-10_2.13-4.0.1.jar
   spark-token-provider-kafka-0-10_2.13-4.0.1.jar


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/03 11:00:37 WARN Utils: Your hostname, Hariprasads-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.11.122.210 instead (on interface en0)
26/08/03 11:00:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/08/03 11:00:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).



spark version: 4.0.1


In [4]:
KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_TOPIC = "healthcare_fhir_stream"

bronze_streaming_path = "../../data_lake/bronze/streaming_data/"
checkpoint_path = "../../data_lake/_checkpoints/streaming_bronze/"

## Message schema

Each Kafka message value is a JSON envelope produced by the producer:

```json
{
  "bundle_resource_type": "Bundle",
  "bundle_type": "transaction",
  "fullUrl": "urn:uuid:...",
  "resource_type": "Encounter",
  "resource": { ...full FHIR resource... },
  "request": {"method": "POST", "url": "Encounter"},
  "source_file_name": "Julio255_Stokes453_....json"
}
```

`resource` is declared as `MapType(String, String)` — the same choice the batch bronze
notebook makes. Every FHIR resource type has a different shape, so a strict schema is
impossible here; the map keeps top-level fields accessible while nested objects and
arrays stay as raw JSON strings for silver to parse.

In [5]:
message_schema = StructType([
    StructField("bundle_resource_type", StringType()),
    StructField("bundle_type",          StringType()),
    StructField("fullUrl",              StringType()),
    StructField("resource_type",        StringType()),
    StructField("resource",             MapType(StringType(), StringType())),
    StructField("request",              MapType(StringType(), StringType())),
    StructField("source_file_name",     StringType()),
])

## Read the stream

`startingOffsets = "earliest"` reads the topic from the beginning **on the first run only**.
After that the checkpoint holds the offsets, and this option is ignored — restarts resume
where they left off.

Kafka gives us `key` / `value` as binary plus metadata columns (`partition`, `offset`,
`timestamp`), so `value` must be cast to string before parsing.

In [6]:
kafka_df = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 50000)   # cap batch size so each micro-batch stays manageable
    .load())

kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## Parse into the bronze schema

`resource_type` is taken from `request.url` — the same field the batch notebook uses,
and cheaper than digging it out of the resource JSON.

The Kafka metadata columns are kept for debugging and replay; the other eight columns
are byte-for-byte the same as batch bronze.

In [7]:
bronze_df = (kafka_df
    .select(
        from_json(col("value").cast("string"), message_schema).alias("m"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp"),
    )
    .select(
        "m.bundle_resource_type",
        "m.bundle_type",
        "m.fullUrl",
        "m.resource",
        "m.request",
        col("m.source_file_name").alias("input_file_name"),
        col("m.request").getItem("url").alias("resource_type"),
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
    )
    .withColumn("ingestion_timestamp", current_timestamp()))

bronze_df.printSchema()

root
 |-- bundle_resource_type: string (nullable = true)
 |-- bundle_type: string (nullable = true)
 |-- fullUrl: string (nullable = true)
 |-- resource: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- request: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- input_file_name: string (nullable = true)
 |-- resource_type: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)



## Write the stream

Two things that have no batch equivalent:

- **`checkpointLocation`** — Spark records which Kafka offsets it has processed here.
  Restarting resumes from the checkpoint instead of reprocessing. Delete this directory
  to force a full re-read from the beginning.
- **`trigger`** — `availableNow=True` processes everything currently in the topic, then
  **stops**. That is what we want for draining the existing backlog. For a live demo
  swap it for `processingTime="30 seconds"`, which keeps running and polls for new data.

`partitionBy("resource_type")` mirrors batch bronze, so silver reads work identically.

In [8]:
query = (bronze_df.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", bronze_streaming_path)
    .option("checkpointLocation", checkpoint_path)
    .partitionBy("resource_type")
    .trigger(availableNow=True)
    .start())

query.awaitTermination()
print("stream finished")
print("last progress:", query.lastProgress)

stream finished
last progress: {
    "id": "7fd4bb74-ae0e-4d27-815d-9e0e3c02ecdd",
    "runId": "df1e2a65-3228-4e9a-ac42-b11a44178b64",
    "name": null,
    "timestamp": "2026-08-03T15:01:02.605Z",
    "batchId": 14,
    "batchDuration": 1050,
    "numInputRows": 12825,
    "inputRowsPerSecond": 11269.771528998244,
    "processedRowsPerSecond": 12214.285714285714,
    "durationMs": {
        "addBatch": 996,
        "commitOffsets": 25,
        "getBatch": 0,
        "latestOffset": 0,
        "queryPlanning": 3,
        "triggerExecution": 1050,
        "walCommit": 25
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[healthcare_fhir_stream]]",
            "startOffset": {
                "healthcare_fhir_stream": {
                    "0": 184715,
                    "1": 256099,
                    "2": 259170
                }
            },
            "endOffset": {
                "healthcare_fhir_stream": {
              

## Verify what landed

In [9]:
written = spark.read.parquet(bronze_streaming_path)
print("total rows:", written.count())

written.groupBy("resource_type").count().orderBy(col("count").desc()).show(30, truncate=False)

total rows: 712809


+------------------------+------+
|resource_type           |count |
+------------------------+------+
|Observation             |274372|
|Procedure               |89778 |
|DiagnosticReport        |64472 |
|ExplanationOfBenefit    |58904 |
|Claim                   |58904 |
|DocumentReference       |33588 |
|Encounter               |33588 |
|MedicationRequest       |25316 |
|Condition               |20153 |
|SupplyDelivery          |13851 |
|MedicationAdministration|10120 |
|Medication              |10120 |
|Immunization            |8088  |
|Device                  |3159  |
|ImagingStudy            |2791  |
|CareTeam                |1944  |
|CarePlan                |1944  |
|Provenance              |574   |
|Patient                 |574   |
|AllergyIntolerance      |569   |
+------------------------+------+



In [10]:
written.select("resource_type", "input_file_name", "kafka_partition", "kafka_offset").show(5, truncate=False)

+--------------------+---------------------------------------------------------------------------+---------------+------------+
|resource_type       |input_file_name                                                            |kafka_partition|kafka_offset|
+--------------------+---------------------------------------------------------------------------+---------------+------------+
|ExplanationOfBenefit|Kathlene786_Parthenia862_Kling921_5aea51c7-02f6-4b0f-7739-5857036cb54d.json|1              |237817      |
|ExplanationOfBenefit|Kathlene786_Parthenia862_Kling921_5aea51c7-02f6-4b0f-7739-5857036cb54d.json|1              |237823      |
|ExplanationOfBenefit|Kathlene786_Parthenia862_Kling921_5aea51c7-02f6-4b0f-7739-5857036cb54d.json|1              |237857      |
|ExplanationOfBenefit|Kathlene786_Parthenia862_Kling921_5aea51c7-02f6-4b0f-7739-5857036cb54d.json|1              |237865      |
|ExplanationOfBenefit|Kathlene786_Parthenia862_Kling921_5aea51c7-02f6-4b0f-7739-5857036cb54d.json|1     

In [11]:
spark.stop()